In [ ]:
# ==============================================================================
# 🚀 DO NOT MODIFY: Standardized Notebook Setup
# ==============================================================================
# This cell is designed to work in both Google Colab and local environments.
# It ensures that the environment is correctly configured by cloning (or
# locating) the project repository and installing the necessary dependencies.
#
# ------------------------------------------------------------------------------
#
#  ⚠️  IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY (NOT ON COLAB):
#
#  This cell will automatically find the repository root and configure your
#  environment. Just make sure you have run: pip install -e .[dev]
#
# ------------------------------------------------------------------------------

import os
import subprocess
import sys
from pathlib import Path

# --- Configuration ---
REPO_URL = "https://github.com/BradSegal/ADH-LLM-Tutorials-2025.git"
REPO_DIR = Path("ADH-LLM-Tutorials-2025")  # The name of the directory once cloned
# --- End of Configuration ---


def find_repo_root(start_path: Path) -> Path | None:
    """
    Find the repository root by looking for pyproject.toml.

    Searches upward from start_path until it finds pyproject.toml or hits root.

    Args:
        start_path: Directory to start searching from.

    Returns:
        Path to repository root, or None if not found.
    """
    current = start_path.resolve()
    while current != current.parent:  # Stop at filesystem root
        if (current / "pyproject.toml").exists():
            return current
        current = current.parent
    return None


def detect_active_branch(repo_dir: Path) -> str:
    """
    Determine the active git branch for pulling updates.

    Tries multiple methods to detect the current branch name.

    Args:
        repo_dir: Path to the git repository.

    Returns:
        Branch name (defaults to 'master' if detection fails).
    """
    commands = [
        "git symbolic-ref --short HEAD",
        "git rev-parse --abbrev-ref HEAD",
    ]
    for cmd in commands:
        result = subprocess.run(
            cmd, shell=True, cwd=repo_dir, capture_output=True, text=True
        )
        if result.returncode == 0:
            branch = result.stdout.strip()
            if branch and not branch.startswith("origin/"):
                return branch
    return "master"


def run_cmd(cmd: str, *, cwd: Path | None = None) -> None:
    """
    Run a shell command and raise an error if it fails.

    Args:
        cmd: The command to run.
        cwd: Optional working directory for the command.

    Raises:
        RuntimeError: If the command returns a non-zero exit code.
    """
    result = subprocess.run(cmd, shell=True, cwd=cwd)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {cmd}")


# --- Detect environment ---
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False


# --- Main setup logic ---
if IN_COLAB:
    print("☁️  Running in Google Colab. Setting up the environment...\n")

    # Determine repository path
    start_dir = Path.cwd()
    if start_dir.name == REPO_DIR.name:
        repo_path = start_dir
    else:
        repo_path = start_dir / REPO_DIR

    # Clone or update repository
    if not repo_path.exists():
        print(f"📥 Cloning repository from {REPO_URL}...")
        run_cmd(f"git clone --quiet {REPO_URL} {repo_path}")
        print(f"✅ Repository cloned to {repo_path}\n")
    else:
        print(f"📂 Repository already exists at {repo_path}")
        active_branch = detect_active_branch(repo_path)
        print(f"🔄 Pulling latest changes from branch '{active_branch}'...")
        run_cmd(f"git pull origin {active_branch} --quiet", cwd=repo_path)
        print(f"✅ Repository updated\n")

    # Verify repository structure
    if not (repo_path / "pyproject.toml").exists():
        raise FileNotFoundError(
            f"Repository structure invalid: pyproject.toml not found in {repo_path}. "
            "The repository may be corrupted."
        )

    # Change working directory and update Python path
    print(f"📁 Changing working directory to {repo_path}")
    os.chdir(repo_path)
    if str(repo_path) not in sys.path:
        sys.path.insert(0, str(repo_path))

    # Install dependencies (smart installation - only installs missing packages)
    from core.notebook.setup import smart_install_dependencies

    result = smart_install_dependencies(
        repo_path=repo_path,
        include_dev=True,
        verbose=True,
    )

    # Fail loudly if critical packages failed to install
    if result["failed"]:
        print(f"\n⚠️  WARNING: {len(result['failed'])} packages failed to install:")
        for pkg in result["failed"]:
            print(f"  - {pkg}")
        print("\nYou may encounter import errors. Please check your internet connection.")

    print("\n" + "=" * 70)
    print("✅ Environment setup complete! You can now proceed with the notebook.")
    print("=" * 70)

else:
    print("💻 Running in local environment. Configuring...\n")

    # Find the repository root
    repo_path = find_repo_root(Path.cwd())

    if repo_path is None:
        raise FileNotFoundError(
            "Could not find repository root (no pyproject.toml found). "
            "Please ensure you are running this notebook from within the "
            "ADH-LLM-Tutorials-2025 repository directory."
        )

    print(f"✅ Found repository root: {repo_path}")

    # Change working directory and update Python path
    print(f"📁 Changing working directory to {repo_path}")
    os.chdir(repo_path)

    if str(repo_path) not in sys.path:
        sys.path.insert(0, str(repo_path))

    print("\n" + "=" * 70)
    print("✅ Local environment configured successfully!")
    print("=" * 70)
    print("\n⚠️  Please ensure you have run: pip install -e .[dev]")
    print("   (Required for local development)")


# 01 - Problem Framing and Exploratory Data Analysis

## Introduction to Sepsis Prediction

**Sepsis** is a life-threatening medical condition that occurs when the body's response to an infection causes widespread inflammation. It can rapidly progress to septic shock, organ failure, and death if not identified and treated early. According to the World Health Organization, sepsis affects over 30 million people worldwide each year and contributes to more than 5 million deaths.

**The Clinical Challenge:**

Early detection of sepsis is critical for improving patient outcomes. Studies show that each hour of delay in treatment increases mortality risk by 4-9%. However, early sepsis detection is challenging because:

- Symptoms are non-specific and can be subtle
- Vital signs change gradually over time
- Clinical presentation varies widely between patients
- Healthcare providers must monitor dozens of patients simultaneously

**Our Goal:**

In this tutorial series, we'll build machine learning models that can predict sepsis onset from time-series physiological data collected in Intensive Care Units (ICUs). We'll use the **PhysioNet Challenge 2019 dataset**, which contains hourly vital signs and laboratory measurements from over 40,000 ICU patients.

By the end of this notebook, you will understand:

1. The structure and characteristics of the sepsis dataset
2. The distribution of key vital signs and demographics
3. The class imbalance challenge (sepsis is relatively rare)
4. How time-series medical data is organized for modeling

Let's begin our exploratory analysis!

In [ ]:
# Import required libraries
import matplotlib.pyplot as plt
import seaborn as sns

from core.data.physionet_sepsis import get_sepsis_data

# Configure visualization settings
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

## Step 1: Load the Sepsis Dataset

We'll use the `get_sepsis_data()` function from our `core` library to load the preprocessed PhysioNet dataset. This function:

- Downloads the raw data from PhysioNet (if not already cached)
- Parses all patient files
- Imputes missing values using forward-fill and median imputation
- Standardizes all physiological features (z-score normalization)
- Returns a clean DataFrame ready for analysis

**Note:** The first time you run this cell, it will download ~1.5 GB of data. Subsequent runs will use the cached version.

In [ ]:
# Load the preprocessed sepsis dataset
df = get_sepsis_data()

# Display basic information about the dataset
print("Dataset Information:")
print("=" * 70)
df.info()

print("\n" + "=" * 70)
print("First Few Rows:")
print("=" * 70)
df.head(10)

## Step 1.5: Preprocessing Transparency - What Happened Behind the Scenes

The `get_sepsis_data()` function didn't just load raw data - it performed several critical preprocessing steps that prepare the data for machine learning. Let's understand what happened:

### 🔧 Preprocessing Steps

#### 1. **Missing Value Imputation**

Medical data often has missing values (lab tests aren't run every hour, sensors can fail, etc.). The preprocessing pipeline used two imputation strategies:

- **Forward-fill**: Copy the last known value forward in time
  - *Example*: If lactate was measured at hour 5 but not hour 6, use hour 5's value for hour 6
  - *Rationale*: Lab values don't change instantly - this is clinically reasonable
  
- **Median imputation**: Fill remaining gaps with the population median
  - *Used for*: Values that have no previous measurement to forward-fill from
  - *Rationale*: Use a "typical" value rather than leaving it undefined

#### 2. **Z-Score Normalization**

All physiological features were standardized using z-score normalization:

```
z = (value - population_mean) / population_std
```

**Why is this necessary?**
- Neural networks learn better when features are on similar scales
- Without normalization: Heart rate (60-100) would dominate temperature (36-38) simply due to magnitude
- With normalization: All features have mean ≈ 0, standard deviation ≈ 1

**How to interpret z-scores:**
- `z = 0`: This value is exactly at the population average
- `z = 2`: This value is 2 standard deviations ABOVE average (abnormally high)
- `z = -2`: This value is 2 standard deviations BELOW average (abnormally low)
- `z > 3` or `z < -3`: Extreme values, potentially clinically significant

**Clinical Example:**
- Original heart rate: 120 bpm
- Population mean: 86 bpm, std: 17 bpm
- Z-score: (120 - 86) / 17 = **2.0** → This patient's heart rate is 2 standard deviations above normal (tachycardia)

### 🔍 Let's Verify the Normalization

If z-score normalization was applied correctly, all feature columns should have:
- **Mean** ≈ 0 (within floating-point precision)
- **Standard deviation** ≈ 1

In [ ]:
# Verify z-score normalization on feature columns
feature_columns_to_check = ["HR", "Temp", "O2Sat", "SBP", "Resp", "BUN", "Lactate"]

print("=" * 70)
print("Verifying Z-Score Normalization")
print("=" * 70)
print(
    "\nIf preprocessing was correct, mean ≈ 0 and std ≈ 1 for all features:\n"
)

stats = df[feature_columns_to_check].describe().loc[["mean", "std"]]
print(stats.round(3))

print("\n" + "=" * 70)
print("✅ Confirmed: All features are z-score normalized!")
print("=" * 70)
print("\n💡 Remember: When you see HR = 2.5 in the data, it means")
print("   'Heart rate is 2.5 standard deviations above the population mean'")
print("   This is more useful for ML than the raw value (e.g., 120 bpm)")

## Step 2: Understanding the Data Structure

The dataset is organized as a **time-series DataFrame** where each row represents one hour of ICU monitoring for a patient. Let's examine the key characteristics:

### Key Columns:

- **`patient_id`**: Unique identifier for each patient (important for grouping time-series)
- **`ICULOS`**: ICU Length of Stay in hours (the timestamp for each measurement)
- **`Age`, `Gender`, `Unit1`, `Unit2`, `HospAdmTime`**: Static patient demographics
- **`HR`, `O2Sat`, `Temp`, `SBP`, `MAP`, `DBP`, `Resp`**: Vital signs measured every hour
- **`BUN`, `Creatinine`, `Glucose`, `Lactate`, etc.**: Laboratory values (measured less frequently)
- **`SepsisLabel`**: Target variable (1 = sepsis detected, 0 = no sepsis)

### Important Notes:

1. **Time-series structure**: Each patient has multiple rows (one per hour in ICU)
2. **Variable length**: Different patients stayed in the ICU for different durations
3. **Feature scaling**: All physiological features have been z-score normalized
4. **No missing values**: The preprocessing pipeline has imputed all missing data

In [ ]:
# Display summary statistics for key columns
print("Summary Statistics:")
print("=" * 70)
key_columns = [
    "Age",
    "Gender",
    "HR",
    "Temp",
    "O2Sat",
    "SBP",
    "Resp",
    "ICULOS",
    "SepsisLabel",
]
df[key_columns].describe()

## Step 2.5: From DataFrames to Tensors - Understanding the Transformation

While we're currently working with a familiar **pandas DataFrame**, sequence models like RNNs and Transformers require data in a different format: **3D tensors**. This is one of the most important conceptual steps that happens "behind the scenes" when we train our models in notebooks 02-04.

### The Transformation Journey

Here's what happens when we convert our DataFrame into model-ready tensors:

```
DataFrame (tabular)
    ↓
Grouped by patient_id
    ↓
Each patient → [sequence_length, num_features] tensor
    ↓
Batch of patients → [batch_size, max_seq_length, num_features] tensor
    ↓
Padding + Masking (handle variable lengths)
```

### Understanding 3D Tensor Structure

When we train a neural network, we process multiple patients simultaneously in **batches**. Each batch becomes a 3D tensor with shape:

- **Dimension 0** (`batch_size`): How many patients in this batch (e.g., 32)
- **Dimension 1** (`sequence_length`): How many hours in ICU (varies per patient!)
- **Dimension 2** (`num_features`): How many measurements per hour (34 features)

**Example**: A batch of 32 patients, where the longest stay is 72 hours, creates a tensor of shape `[32, 72, 34]`.

### The Padding Problem

**Challenge**: Not all patients stay in the ICU for the same duration. One patient might have 24 hours of data, another might have 72 hours.

**Solution**: We **pad** shorter sequences with zeros to match the longest sequence in the batch.

**Example**:
- Patient A: 24 hours → padded to 72 with 48 hours of zeros
- Patient B: 72 hours → no padding needed
- Patient C: 36 hours → padded to 72 with 36 hours of zeros

### The Masking Solution

**Problem**: We don't want the model to learn from padding (it's not real data!).

**Solution**: We create a **mask** - a boolean array that marks which timesteps are real vs. padded:

```
Patient A mask: [True, True, ..., True (×24), False, False, ..., False (×48)]
                 └─ Real hours ─┘  └───── Padded hours ─────┘
```

The model uses this mask to:
1. Ignore padded values during training
2. Only compute loss on real timesteps
3. Extract predictions from the last **real** hour (not padding)

### Why This Matters

Understanding this transformation is crucial because:

1. **It explains why we group by `patient_id`**: Each patient is one sequence
2. **It explains why `ICULOS` matters**: It's the timestamp for ordering the sequence
3. **It explains variable-length handling**: Padding + masking solves the problem
4. **It sets up what you'll see in notebooks 02-04**: The `create_dataloaders()` function handles all of this automatically!

In the next few cells, let's see this transformation in action for a single patient.

In [ ]:
# Let's examine the transformation for a single patient
sample_patient_id = df["patient_id"].iloc[0]
patient_data = df[df["patient_id"] == sample_patient_id].sort_values("ICULOS")

# Get feature columns (exclude patient_id, ICULOS, and SepsisLabel)
feature_columns = [
    col
    for col in df.columns
    if col not in ["patient_id", "ICULOS", "SepsisLabel"]
]

print("=" * 70)
print(f"📊 Patient {sample_patient_id} - Transformation Example")
print("=" * 70)
print(f"\n1️⃣  DataFrame Representation:")
print(f"   - Number of rows (hours in ICU): {len(patient_data)}")
print(f"   - Number of feature columns: {len(feature_columns)}")
print(f"   - First few rows:\n")
print(patient_data[["ICULOS", "HR", "Temp", "O2Sat", "SepsisLabel"]].head())

print(f"\n2️⃣  Tensor Representation (after conversion):")
print(f"   - Shape: [{len(patient_data)}, {len(feature_columns)}]")
print(f"   - Interpretation: {len(patient_data)} timesteps × {len(feature_columns)} features")

print(f"\n3️⃣  If batched with other patients (batch_size=32):")
# Find the longest sequence in the dataset
max_seq_length = df.groupby("patient_id").size().max()
print(f"   - Longest patient stay in dataset: {max_seq_length} hours")
print(f"   - Batch tensor shape: [32, {max_seq_length}, {len(feature_columns)}]")
print(f"   - This patient needs {max_seq_length - len(patient_data)} hours of padding")

print(f"\n4️⃣  Padding and Masking:")
print(f"   - Real timesteps: hours 1-{len(patient_data)} → mask = True")
print(f"   - Padded timesteps: hours {len(patient_data)+1}-{max_seq_length} → mask = False")
print(f"   - The model will ONLY learn from the {len(patient_data)} real hours!")

print("\n" + "=" * 70)
print("✅ This transformation happens automatically in `create_dataloaders()`")
print("=" * 70)

## Step 3: Class Imbalance Analysis

One of the key challenges in sepsis prediction is **class imbalance**. Sepsis is relatively rare, even in ICU populations. Let's examine the distribution of positive and negative cases.

In [ ]:
# Calculate class distribution
sepsis_counts = df["SepsisLabel"].value_counts()
sepsis_percentages = df["SepsisLabel"].value_counts(normalize=True) * 100

print("Sepsis Label Distribution:")
print("=" * 70)
print(f"No Sepsis (0): {sepsis_counts[0]:,} records ({sepsis_percentages[0]:.2f}%)")
print(f"Sepsis (1):    {sepsis_counts[1]:,} records ({sepsis_percentages[1]:.2f}%)")
print(f"\nImbalance Ratio: {sepsis_counts[0] / sepsis_counts[1]:.2f}:1")

# Visualize class distribution
fig, ax = plt.subplots(figsize=(8, 6))
sepsis_counts.plot(kind="bar", ax=ax, color=["steelblue", "coral"])
ax.set_title("Distribution of Sepsis Labels", fontsize=14, fontweight="bold")
ax.set_xlabel("Sepsis Label", fontsize=12)
ax.set_ylabel("Number of Records", fontsize=12)
ax.set_xticklabels(["No Sepsis (0)", "Sepsis (1)"], rotation=0)
ax.grid(axis="y", alpha=0.3)

# Add count labels on top of bars
for i, count in enumerate(sepsis_counts):
    ax.text(i, count + 10000, f"{count:,}", ha="center", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()

print("\n⚠️  Key Observation:")
print("The dataset exhibits significant class imbalance. This is clinically realistic")
print("but presents a modeling challenge. We'll need to carefully design our training")
print("and evaluation strategy to account for this imbalance.")

## Step 4: Demographic Analysis

Let's examine the distribution of patient demographics to understand our study population.

In [ ]:
# Analyze demographics (using one row per patient)
patient_demographics = df.groupby("patient_id").first()[["Age", "Gender"]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age distribution
axes[0].hist(
    patient_demographics["Age"],
    bins=30,
    color="steelblue",
    edgecolor="black",
    alpha=0.7,
)
axes[0].set_title("Age Distribution of ICU Patients", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Age (years)", fontsize=11)
axes[0].set_ylabel("Number of Patients", fontsize=11)
axes[0].axvline(
    patient_demographics["Age"].median(),
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Median: {patient_demographics['Age'].median():.1f} years",
)
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

# Gender distribution
gender_counts = patient_demographics["Gender"].value_counts()
axes[1].bar(
    gender_counts.index,
    gender_counts.values,
    color=["coral", "lightblue"],
    edgecolor="black",
)
axes[1].set_title("Gender Distribution of ICU Patients", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Gender (0=Female, 1=Male)", fontsize=11)
axes[1].set_ylabel("Number of Patients", fontsize=11)
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(["Female (0)", "Male (1)"])
axes[1].grid(axis="y", alpha=0.3)

# Add count labels
for i, count in enumerate(gender_counts.values):
    axes[1].text(
        i, count + 50, f"{count:,}", ha="center", fontsize=11, fontweight="bold"
    )

plt.tight_layout()
plt.show()

print(f"Total number of unique patients: {len(patient_demographics):,}")
print(f"Average age: {patient_demographics['Age'].mean():.1f} years")
age_min = patient_demographics["Age"].min()
age_max = patient_demographics["Age"].max()
print(f"Age range: {age_min:.0f} - {age_max:.0f} years")

## Understanding Sequence Models: Why Do We Need Them for Sepsis?

Before we continue with the exploratory analysis, let's address a fundamental question: **Why do we need sequence models for this problem?** Why can't we just use a simple classifier that looks at one hour of data?

### The Clinical Reality: Sepsis is a Trajectory, Not a Snapshot

Sepsis doesn't happen suddenly - it's a **progressive deterioration** over time. Early warning signs (subtle increases in heart rate, slight fever, rising lactate) can appear hours before a clinical diagnosis.

**Example Timeline:**
- **Hour 1**: Normal vital signs, patient stable
- **Hour 5**: Slight temperature elevation (37.8°C), HR 95 → might be normal variation
- **Hour 10**: Temperature 38.5°C, HR 105, lactate rising → pattern emerging
- **Hour 15**: Temperature 39°C, HR 115, BP dropping → sepsis developing

A model that only looks at **one hour** would miss the trajectory. But a **sequence model** can learn:
- "Temperature has been rising steadily for 10 hours" → higher risk
- "Heart rate spiked suddenly after being stable" → concerning pattern
- "Lactate increased while BP decreased" → multi-variable temporal pattern

### Three Approaches to Sequence Modeling

In notebooks 02-04, you'll train three different sequence model architectures. Here's a preview of how they work:

#### 1️⃣ **GRU (Gated Recurrent Unit)** - Notebook 02

**How it processes sequences:**
```
Hour 1 → [GRU] → hidden state h₁
              ↓
Hour 2 → [GRU] → hidden state h₂  (remembers h₁)
              ↓
Hour 3 → [GRU] → hidden state h₃  (remembers h₂)
              ↓
  ...
              ↓
Hour 24 → [GRU] → final hidden state → [Classifier] → Prediction
```

**Key concept**: The GRU maintains a "memory" (hidden state) that gets updated at each hour. It decides what to remember and what to forget using **gates**.

**Strength**: Efficient, good baseline for sequential data  
**Limitation**: Can struggle with very long sequences (information from hour 1 may "fade" by hour 50)

#### 2️⃣ **LSTM (Long Short-Term Memory)** - Notebook 03

**How it's different from GRU:**
- Maintains TWO memory components: **cell state** (long-term memory) + **hidden state** (short-term memory)
- Uses three gates (forget, input, output) instead of two
- Better at preserving information over longer time periods

**Key concept**: The separate cell state acts like a "conveyor belt" that can carry important information (e.g., "patient had high lactate at hour 5") all the way to the final prediction.

**Strength**: Better at capturing long-term dependencies than GRU  
**Limitation**: Slightly more complex and slower to train

#### 3️⃣ **Transformer** - Notebook 04

**How it's fundamentally different:**
```
[Hour 1, Hour 2, Hour 3, ..., Hour 24] → All processed in parallel
                    ↓
            Self-Attention Layer
         (Each hour "looks at" all other hours)
                    ↓
            Attention Weights
         (Learn which hours are important)
                    ↓
              Mean Pooling → [Classifier] → Prediction
```

**Key concept**: Instead of processing sequentially (hour → hour → hour), Transformers use **self-attention** to let every hour directly "attend to" every other hour.

**Example**: When predicting at hour 24, the model might learn:
- "Hour 5's lactate spike is highly relevant" (high attention weight)
- "Hours 12-18 vital signs are moderately relevant" (medium attention weight)
- "Hour 2 is not predictive" (low attention weight)

**Strength**: Can capture complex, long-range relationships; parallel processing  
**Limitation**: More computationally expensive, requires more data to train

### Comparing the Architectures

| Aspect | GRU | LSTM | Transformer |
|--------|-----|------|-------------|
| **Processing** | Sequential (hour-by-hour) | Sequential (hour-by-hour) | Parallel (all hours at once) |
| **Memory** | Single hidden state | Cell state + hidden state | Attention mechanism |
| **Long-term deps** | Moderate | Better | Best |
| **Speed** | Fast | Medium | Slower (attention is expensive) |
| **Complexity** | Simpler | More complex | Most complex |

### What You'll Do in Notebooks 02-04

Each notebook follows the same high-level pattern:

```python
# 1. Load configuration (defines hyperparameters)
config = load_config("configs/gru.yaml")

# 2. Create data loaders (handles batching, padding, masking)
train_loader, val_loader = create_dataloaders(train_config=config, df=df)

# 3. Initialize model
model = GRUModel(config)  # or LSTMModel, or TransformerModel

# 4. Train the model
trainer = Trainer(model, train_loader, val_loader, config)
history = trainer.fit()  # This does all the hard work!

# 5. Visualize training curves
plot_training_history(history)
```

**Important**: The `core` library handles all the complexity (PyTorch training loops, gradient computation, backpropagation, optimization). Your job is to:
- Understand the **high-level architectural differences** (above)
- Run the training pipeline
- Compare model performance in notebook 05

### Why Abstract Away the Complexity?

You might wonder: "Why don't we implement the training loop ourselves?"

**Answer**: The goal of this tutorial is to teach you:
1. How to think about sequence modeling for clinical prediction
2. How to compare different architectures (GRU vs. LSTM vs. Transformer)
3. How to evaluate and explain model predictions

If we made you write PyTorch training loops from scratch, you'd spend hours debugging CUDA errors and gradient shapes instead of learning about sepsis prediction!

The abstraction is **intentional and pedagogical**. In notebook 05, we'll "reveal the magic" and explain what `trainer.fit()` actually does under the hood.

### Reflection Questions

Before moving to the next section, think about:

1. **Why might a Transformer outperform a GRU for sepsis prediction?**  
   *Hint: Think about which hours might be most important for prediction*

2. **What's a potential downside of using a Transformer instead of GRU?**  
   *Hint: Think about computational cost and inference speed in a real ICU*

3. **If you only had data up to hour 6, which model architecture might you prefer?**  
   *Hint: Think about how much data Transformers need to learn attention patterns*

Let's continue exploring the data to build intuition about these temporal patterns!

## Step 5: Vital Signs Analysis

Let's examine the distribution of key vital signs across all time points. Remember that these values have been **z-score normalized**, so they represent deviations from the population mean in units of standard deviations.

In [ ]:
# Plot distributions of key vital signs (z-score normalized)
vital_signs = ["HR", "Temp", "O2Sat", "SBP", "Resp"]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, vital in enumerate(vital_signs):
    axes[i].hist(df[vital], bins=50, color="steelblue", edgecolor="black", alpha=0.7)
    axes[i].set_title(
        f"{vital} Distribution (Z-Score Normalized)", fontsize=12, fontweight="bold"
    )
    axes[i].set_xlabel(f"{vital} (standard deviations)", fontsize=10)
    axes[i].set_ylabel("Frequency", fontsize=10)
    axes[i].axvline(0, color="red", linestyle="--", linewidth=2, label="Mean = 0")
    axes[i].legend()
    axes[i].grid(axis="y", alpha=0.3)

# Remove extra subplot
fig.delaxes(axes[5])

plt.tight_layout()
plt.show()

print("\n📊 Interpretation:")
print("Since all features are z-score normalized, they should be centered around 0")
print("with a standard deviation close to 1. Extreme values (far from 0) represent")
print("physiological abnormalities that may be clinically significant.")

## Step 6: Time-Series Visualization

To understand how vital signs evolve over time, let's visualize the time-series for a few example patients. We'll select one patient with sepsis and one without to compare their trajectories.

In [ ]:
# Select one patient with sepsis and one without
sepsis_patients = df[df["SepsisLabel"] == 1]["patient_id"].unique()
no_sepsis_patients = df[df["SepsisLabel"] == 0]["patient_id"].unique()

# Choose the first patient from each group (you can change these indices)
patient_with_sepsis = sepsis_patients[0]
patient_without_sepsis = no_sepsis_patients[0]

# Extract time-series for both patients
sepsis_timeseries = df[df["patient_id"] == patient_with_sepsis].sort_values("ICULOS")
no_sepsis_timeseries = df[df["patient_id"] == patient_without_sepsis].sort_values(
    "ICULOS"
)

# Plot vital signs over time
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
vitals_to_plot = ["HR", "Temp", "O2Sat", "SBP"]

for i, vital in enumerate(vitals_to_plot):
    row, col = i // 2, i % 2
    ax = axes[row, col]

    # Plot both patients
    ax.plot(
        sepsis_timeseries["ICULOS"],
        sepsis_timeseries[vital],
        marker="o",
        label=f"Patient {patient_with_sepsis} (Sepsis)",
        color="coral",
        linewidth=2,
    )
    ax.plot(
        no_sepsis_timeseries["ICULOS"],
        no_sepsis_timeseries[vital],
        marker="s",
        label=f"Patient {patient_without_sepsis} (No Sepsis)",
        color="steelblue",
        linewidth=2,
    )

    ax.set_title(f"{vital} Over Time", fontsize=12, fontweight="bold")
    ax.set_xlabel("ICU Length of Stay (hours)", fontsize=10)
    ax.set_ylabel(f"{vital} (z-score)", fontsize=10)
    ax.legend(loc="best")
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color="gray", linestyle="--", linewidth=1, alpha=0.5)

plt.tight_layout()
plt.show()

print("\n🔍 Key Observation:")
print(
    "Notice how vital signs can vary significantly over time for individual patients."
)
print("The goal of our sequence models is to learn patterns in these")
print("temporal trajectories that are predictive of sepsis onset.")

## Step 7: Looking Ahead - Your Modeling Journey

Now that you understand the data, let's preview the path ahead. In notebooks 02-04, you'll build and train three sequence models. Here's what to expect:

### The Standard Workflow (All Three Notebooks)

Each modeling notebook (GRU, LSTM, Transformer) follows this pattern:

```python
# Step 1: Load configuration
config = yaml.safe_load(Path("configs/gru.yaml").read_text())
model_config = GRUConfig(**config["model"])
train_config = TrainConfig(**config["training"])

# Step 2: Create data loaders
train_loader, val_loader = create_dataloaders(train_config=train_config, df=df)

# Step 3: Initialize model
model = GRUModel(model_config)

# Step 4: Train
trainer = Trainer(model, train_loader, val_loader, train_config, save_path="models/gru.pt")
history, best_model_path = trainer.fit()

# Step 5: Visualize
plot_training_history(history)
```

**That's it!** About 10-15 lines of code to train a state-of-the-art sequence model.

### What's Hidden: The `core` Library's Heavy Lifting

While you write ~10 lines of code, the `core` library is doing ~1,000+ lines of work:

**1. Data Pipeline** (`core/data/loaders.py` - 241 lines)
- Converts DataFrame → PyTorch tensors
- Groups patients into sequences
- Handles variable-length sequences with padding
- Creates boolean masks to track real vs. padded timesteps
- Batches patients together (32 at a time)
- Shuffles training data, keeps validation deterministic

**2. Model Architectures** (`core/models/*.py` - ~165 lines each)
- **GRU**: Multi-layer recurrent network with dropout and hidden state extraction
- **LSTM**: Dual-state (cell + hidden) recurrent network
- **Transformer**: Positional encoding + multi-head self-attention + mean pooling

**3. Training Loop** (`core/train/trainer.py` - 549 lines)
- Device management (CUDA if available, else CPU)
- For each epoch (20 total):
  - **Training phase**: Forward pass → compute loss → backward pass → update weights
  - **Validation phase**: Forward pass → compute loss (no weight updates)
- Gradient clipping (prevents exploding gradients)
- Best model checkpointing (saves model with lowest validation loss)
- Progress bars and logging
- Training history tracking

**4. Evaluation** (`core/evaluation/` - 200+ lines)
- Model inference (generate predictions)
- Compute AUROC and AUPRC metrics
- ROC curve generation

### Why This Abstraction?

**The pedagogical goal** is for you to:
- ✅ Understand the **conceptual differences** between GRU, LSTM, and Transformer
- ✅ Learn how to **configure, train, and evaluate** sequence models
- ✅ **Compare model performance** and make informed architecture choices
- ✅ **Explain model predictions** using attribution methods

**NOT** to:
- ❌ Debug PyTorch tensor shapes and CUDA errors
- ❌ Implement backpropagation from scratch
- ❌ Write custom data collation functions

The complexity is abstracted so you can focus on the **science** (does LSTM beat GRU?) rather than the **engineering** (how do I pad tensors?).

### What You'll Learn in Each Notebook

| Notebook | Model | Key Concept | What You'll See |
|----------|-------|-------------|-----------------|
| **02 - RNN Baseline** | GRU | Sequential processing with hidden state | Training from scratch, loss curves, baseline performance |
| **03 - LSTM** | LSTM | Dual-state memory (cell + hidden) | Comparing GRU vs. LSTM, longer-term dependencies |
| **04 - Transformer** | Transformer | Self-attention mechanism | Parallel processing, positional encoding, attention |
| **05 - Compare & Explain** | All three | Rigorous evaluation + explainability | AUROC/AUPRC comparison, Integrated Gradients, feature importance |

### Hyperparameters You'll Encounter

When you load the YAML configs, you'll see these key hyperparameters:

```yaml
model:
  input_size: 34          # Number of features
  hidden_size: 64         # Size of hidden state (bigger = more capacity)
  num_layers: 2           # Stack multiple GRU/LSTM/Transformer layers
  dropout: 0.2            # Regularization (randomly drop 20% of connections)

training:
  batch_size: 32          # Process 32 patients at once
  epochs: 20              # Complete passes through training data
  learning_rate: 0.001    # How fast the model learns (Adam optimizer)
```

**Don't worry** - you don't need to understand all of these deeply right now. We'll explain them in notebook 05 when we compare models.

### What Happens When You Run `trainer.fit()`?

This single line of code triggers:

1. **Epoch Loop**: Repeat 20 times
2. **Batch Loop**: For each batch of 32 patients:
   - Load padded sequences + masks from DataLoader
   - **Forward Pass**: Run model, get predictions
   - **Compute Loss**: Binary cross-entropy between predictions and labels
   - **Backward Pass**: Compute gradients (how to improve)
   - **Optimizer Step**: Update model weights using gradients
   - **Gradient Clipping**: Prevent gradients from exploding
3. **Validation Loop**: Evaluate on validation set (no weight updates)
4. **Checkpointing**: Save model if validation loss improved
5. **Return**: Training history (loss curves) + path to best model

**In notebook 05**, we'll "open the hood" and explain each of these steps in detail.

### A Note on Training Time

- **GRU**: ~2-3 minutes on CPU, ~30 seconds on GPU
- **LSTM**: ~2-3 minutes on CPU, ~30 seconds on GPU  
- **Transformer**: ~5-7 minutes on CPU, ~1 minute on GPU

In Google Colab (free tier), you can enable GPU acceleration:
`Runtime → Change runtime type → Hardware accelerator → GPU`

### Ready to Build Models?

You now have:
- ✅ Understanding of the sepsis dataset
- ✅ Knowledge of the temporal structure
- ✅ Awareness of the class imbalance challenge
- ✅ Preview of the three model architectures
- ✅ Expectation of the training workflow

**Next stop**: Notebook 02 - Building your first GRU baseline!

## Conclusion and Key Takeaways

In this notebook, we explored the PhysioNet 2019 Sepsis Challenge dataset and gained important insights:

### Key Findings:

1. **Dataset Structure**: We have a time-series dataset with variable-length patient stays (multiple hourly measurements per patient)

2. **Significant Class Imbalance**: Sepsis cases are relatively rare, creating a challenging but realistic prediction problem

3. **Rich Feature Set**: 34 physiological features including vital signs and laboratory values

4. **Clean Data**: The preprocessing pipeline has handled missing values and standardized all features

5. **Temporal Patterns**: Vital signs show clear temporal evolution, motivating the use of sequence models

### Next Steps:

Now that we understand our data, we're ready to build our first predictive model! In the next notebook (`02_rnn_baseline.ipynb`), we will:

- Implement a **GRU (Gated Recurrent Unit)** baseline model
- Learn the standard training workflow for sequence models
- Evaluate model performance on held-out validation data
- Establish a baseline for comparison with more advanced models

Let's get started with modeling!